# LangChain Data Analysis Agent

Free stack: Groq API + LangChain + pandas.
Get a free API key at https://console.groq.com/keys

### 1. Install

In [ ]:
!pip install -q langchain langchain-groq langchain-experimental pandas matplotlib tabulate

### 2. API key

In [ ]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Groq API key: ")

### 3. Sample data

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 500

cities = ["Karachi", "Lahore", "Islamabad", "Sukkur", "Multan"]
categories = ["Laptop", "Phone", "Tablet", "Headphones", "Smartwatch"]
channels = ["Online", "Retail Store", "Reseller"]

prices = {
    "Laptop": 850,
    "Phone": 420,
    "Tablet": 300,
    "Headphones": 60,
    "Smartwatch": 150,
}

df = pd.DataFrame({
    "order_id": range(10001, 10001 + n),
    "date": pd.date_range("2024-01-01", periods=n, freq="D"),
    "city": np.random.choice(cities, n),
    "category": np.random.choice(categories, n),
    "channel": np.random.choice(channels, n, p=[0.5, 0.3, 0.2]),
    "units": np.random.randint(1, 15, n),
    "customer_age": np.random.randint(18, 65, n),
    "rating": np.round(np.random.uniform(2.0, 5.0, n), 1),
    "returned": np.random.choice([0, 1], n, p=[0.9, 0.1]),
})

df["unit_price"] = df["category"].map(prices)
df["discount_pct"] = np.random.choice([0, 5, 10, 15, 20], n)
df["revenue"] = (df["units"] * df["unit_price"] * (1 - df["discount_pct"] / 100)).round(2)
df["month"] = df["date"].dt.strftime("%Y-%m")

df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe()

### 4. Build the agent

In [ ]:
from langchain_groq import ChatGroq
from langchain_experimental.agents import create_pandas_dataframe_agent

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
)

prefix = """You are working with a pandas dataframe in Python named `df`.
The full dataset is ALREADY loaded in memory with all rows.
CRITICAL INSTRUCTION:
DO NOT redefine, recreate, or mock `df` (e.g. NEVER run `df = pd.DataFrame(...)`).
Always perform your calculations directly on the existing `df` variable.
"""

agent = create_pandas_dataframe_agent(
    llm,
    df,
    prefix=prefix,
    verbose=True,
    allow_dangerous_code=True,
    agent_type="tool-calling",
    max_iterations=10,
)


### 5. Helper function

In [ ]:
def ask(question):
    result = agent.invoke(question)
    output = result["output"]
    print("\n" + "-" * 70)
    print(output)
    return output


### 6. Basic queries

In [ ]:
ask("How many rows and columns? List the column names and their data types.")

In [ ]:
ask("What is the total revenue across all orders?")

In [ ]:
ask("Which city has the highest total revenue? Show the full ranking with numbers.")

In [ ]:
ask("What is the average rating per category, sorted highest to lowest?")

### 7. Grouping and aggregation

In [ ]:
ask("Show total revenue and total units for each channel.")

In [ ]:
ask("Which month had the highest revenue? Show all months sorted by revenue.")

In [ ]:
ask("What is the return rate per category? Show it as a percentage.")

### 8. Multi-step reasoning

In [ ]:
ask("""Compare the Online and Retail Store channels.
For each one show total revenue, average order value, average rating, and return rate.
Then state which channel performs better and why.""")

In [ ]:
ask("""Find the top 3 city and category combinations by revenue.
For each show revenue, units sold, and average discount percentage.""")

In [ ]:
ask("""Split customers into age groups: 18-29, 30-44, and 45-64.
For each group show total revenue, average rating, and most purchased category.""")

### 9. Charts

In [ ]:
import matplotlib.pyplot as plt

ask("""Create a bar chart of total revenue by city.
Add a title and axis labels. Call plt.show() at the end.""")

In [ ]:
ask("""Create a line chart of monthly revenue over time.
Rotate the x labels 45 degrees, add a title and axis labels.
Call plt.show() at the end.""")

In [ ]:
ask("""Create a grouped bar chart of revenue by category split by channel.
Add a legend, title, and axis labels. Call plt.show() at the end.""")

In [ ]:
ask("""Create a histogram of customer ratings using 15 bins.
Add a title and axis labels. Call plt.show() at the end.""")

### 10. Insights

In [ ]:
ask("""Act as a business analyst.
Find 3 insights from this data that are not immediately obvious.
For each give the finding, the supporting numbers, and one recommendation.""")

In [ ]:
ask("""Does discount level affect the return rate?
Group by discount_pct, show the return rate for each level, and state your conclusion.""")

### 11. Use your own CSV

In [ ]:
# Upload a CSV with the file panel on the left, then uncomment and run.

# df = pd.read_csv("/content/your_file.csv")
#
# agent = create_pandas_dataframe_agent(
#     llm,
#     df,
#     verbose=True,
#     allow_dangerous_code=True,
#     agent_type="tool-calling",
#     max_iterations=10,
# )
#
# ask("Describe this dataset and suggest questions I could ask about it.")

### 12. Your own questions

In [ ]:
ask("")

### Notes

- `allow_dangerous_code=True` executes generated Python in this runtime. Use only with data you can afford to lose.
- Chart not appearing: add "call plt.show()" to the question.
- Agent looping: make the question more specific or raise `max_iterations`.
- Rate limit error: wait one minute and rerun.
- To verify an answer, ask: "show me the pandas code you used".
- To run offline with Ollama, replace the llm line with:

```python
from langchain_ollama import ChatOllama
llm = ChatOllama(model="llama3.1")
```